# Model Optimization: Knowledge Distillation

In this notebook, we'll apply knowledge distillation to our models using distributed processing. Knowledge distillation is a technique where a smaller "student" model is trained to mimic the behavior of a larger "teacher" model.

## What is Knowledge Distillation?

Knowledge distillation is a model compression technique where a small model (student) is trained to mimic a larger, more complex model (teacher). The key insight is that the teacher model's outputs contain rich information beyond just the predicted class - they contain the relative probabilities across all classes, which represent the teacher's "dark knowledge".

### How Knowledge Distillation Works:

1. **Teacher Model**: A large, pre-trained model with high accuracy but high computational requirements
2. **Student Model**: A smaller model architecture that we want to train
3. **Distillation Process**: The student is trained using a combination of:
   - **Hard Targets**: The actual ground truth labels (standard supervised learning)
   - **Soft Targets**: The probability distributions output by the teacher model

### Benefits of Knowledge Distillation:
- **Reduced Model Size**: Student models are typically much smaller than teacher models
- **Faster Inference**: Smaller models require less computation for predictions
- **Lower Memory Requirements**: Smaller models use less memory during inference
- **Preserved Accuracy**: Student models often retain much of the teacher's performance

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform knowledge distillation on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
# Import required libraries
import os
import sys
import time
import json
import boto3
import sagemaker
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import specific modules for this notebook
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
from IPython.display import clear_output

## 2. Load Workshop Settings

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")

# Initialize S3 client
s3_client = boto3.client('s3')

## 3. Load Model Information

In [ ]:
# Load model information from previous notebooks
try:
    with open('model_info.json', 'r') as f:
        model_info_dict = json.load(f)
    print(f"Loaded model information for {len(model_info_dict)} models")
except FileNotFoundError:
    print("model_info.json not found. Creating default model info.")
    model_info_dict = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased"
        }
    }
    
    # Save model info to file
    with open('model_info.json', 'w') as f:
        json.dump(model_info_dict, f, indent=2)
    print("Created default model info with sentiment analysis model")

# Display model info
for model_key, info in model_info_dict.items():
    print(f"\nModel: {model_key}")
    print(f"  Name: {info['model_name']}")
    print(f"  Task: {info['task']}")
    print(f"  S3 URI: {info.get('s3_uri', 'Not available')}")

# Filter models suitable for distillation (text classification models)
distillable_models = {k: v for k, v in model_info_dict.items() 
                     if v.get('task') == 'text-classification'}

print(f"\nFound {len(distillable_models)} models suitable for knowledge distillation:")
for model_key in distillable_models.keys():
    print(f"  - {model_key}")

## 4. Configure Knowledge Distillation Jobs

Knowledge distillation works by training a smaller "student" model to mimic a larger "teacher" model. We'll configure SageMaker Processing jobs to perform this distillation process.

In [ ]:
# Load workshop configuration
with open('workshop_config.json', 'r') as f:
    workshop_config = json.load(f)

# Set up SageMaker session
sagemaker_session = sagemaker.Session()
role = workshop_config['role']
region = workshop_config['region']
bucket = workshop_config['s3_bucket']
prefix = workshop_config['s3_prefix']

print(f"SageMaker session established in region: {region}")
print(f"Using S3 bucket: {bucket}")
print(f"Using S3 prefix: {prefix}")

# Create PyTorch processor for distillation
processor = PyTorchProcessor(
    framework_version='2.6.0',
    py_version='py312',
    role=role,
    instance_type=OPTIMIZATION_INSTANCE_TYPE,
    instance_count=1,
    base_job_name='knowledge-distillation',
    sagemaker_session=sagemaker_session
)

print(f"Created PyTorch processor with instance type: {OPTIMIZATION_INSTANCE_TYPE}")

## 5. Define Student Model Architectures

We'll define smaller student architectures that will learn from the teacher models.

In [ ]:
# Define student model architectures
student_architectures = {
    "distilbert-small": {
        "model_name": "distilbert-base-uncased",
        "num_hidden_layers": 3,
        "hidden_size": 384,
        "description": "Small DistilBERT with 3 layers and 384 hidden size"
    },
    "distilbert-tiny": {
        "model_name": "distilbert-base-uncased",
        "num_hidden_layers": 2,
        "hidden_size": 256,
        "description": "Tiny DistilBERT with 2 layers and 256 hidden size"
    }
}

print("Available student architectures:")
for arch_name, config in student_architectures.items():
    print(f"  {arch_name}: {config['description']}")

# Create student info files for each architecture
for arch_name, config in student_architectures.items():
    student_info = {arch_name: config}
    
    filename = f"student_{arch_name}_info.json"
    with open(filename, 'w') as f:
        json.dump(student_info, f, indent=2)
    
    print(f"Created {filename}")

# Upload student info files to S3
for arch_name in student_architectures.keys():
    filename = f"student_{arch_name}_info.json"
    s3_key = f"{prefix}/distillation/student_configs/{filename}"
    
    s3_client.upload_file(filename, bucket, s3_key)
    print(f"Uploaded {filename} to s3://{bucket}/{s3_key}")

## 6. Launch Knowledge Distillation Jobs

Now we'll launch distillation jobs for each teacher-student pair.

In [ ]:
# Launch distillation jobs for all suitable models in parallel
job_names = []  # List to store all job names
job_output_paths = {}

# First, prepare all the job configurations
job_configs = {}
print("Preparing distillation jobs for suitable models...")

for model_key in distillable_models.keys():
    model_info = distillable_models[model_key]
    
    # Create teacher info file for this model
    teacher_info = {model_key: model_info}
    teacher_filename = f"teacher_{model_key}_info.json"
    
    with open(teacher_filename, 'w') as f:
        json.dump(teacher_info, f, indent=2)
    
    # Upload teacher info to S3
    teacher_s3_key = f"{prefix}/distillation/teacher_configs/{teacher_filename}"
    s3_client.upload_file(teacher_filename, bucket, teacher_s3_key)
    teacher_s3_uri = f"s3://{bucket}/{teacher_s3_key}"
    
    # Create jobs for each student architecture
    for arch_name in student_architectures.keys():
        student_filename = f"student_{arch_name}_info.json"
        student_s3_key = f"{prefix}/distillation/student_configs/{student_filename}"
        student_s3_uri = f"s3://{bucket}/{student_s3_key}"
        
        # Define the output path
        job_key = f"{model_key}-{arch_name}"
        output_path = f's3://{bucket}/optimization/outputs/{job_key}-distilled'
        job_output_paths[job_key] = output_path
        
        # Define inputs and outputs
        inputs = [
            ProcessingInput(
                source=teacher_s3_uri,
                destination='/opt/ml/processing/input/teacher'
            ),
            ProcessingInput(
                source=student_s3_uri,
                destination='/opt/ml/processing/input/student'
            )
        ]
        
        outputs = [
            ProcessingOutput(
                output_name='distilled-model',
                source='/opt/ml/processing/output',
                destination=output_path
            )
        ]
        
        # Store the job configuration
        job_configs[job_key] = {
            'inputs': inputs,
            'outputs': outputs,
            'arguments': [
                '--teacher-info-path', '/opt/ml/processing/input/teacher/' + teacher_filename,
                '--student-info-path', '/opt/ml/processing/input/student/' + student_filename,
                '--temperature', '2.0',
                '--alpha', '0.5',
                '--epochs', '3',
                '--batch-size', '16'
            ],
            'output_path': output_path
        }
        print(f"Prepared job configuration for {job_key}")

# Now launch all jobs in parallel
print("\nLaunching distillation jobs in parallel...")

# Check if we have any models to distill
if len(job_configs) == 0:
    print("No suitable models to distill. Skipping job launch.")
else:
    for job_key, config in job_configs.items():
        try:
            # Create a unique job name with timestamp to avoid conflicts
            timestamp = int(time.time())
            job_name = f"distillation-{job_key}-{timestamp}"
            
            # Run the processing job with the unique name
            processor.run(
                code='distillation_script.py',
                source_dir='distillation_scripts',
                inputs=config['inputs'],
                outputs=config['outputs'],
                arguments=config['arguments'],
                wait=False,
                job_name=job_name
            )
            
            # Store the job name for tracking
            job_names.append(job_name)
            print(f"Launched job for {job_key}: {job_name}")
        except Exception as e:
            print(f"Error launching job for {job_key}: {e}")

    print("\nAll jobs launched. You can monitor their progress in the SageMaker console.")

## 7. Monitor Job Progress

Let's check the status of our distillation jobs.

In [ ]:
# Monitor job progress with continuous polling until completion
if job_names:
    print(f"Monitoring {len(job_names)} distillation jobs...")
    print("Jobs will be monitored until completion. This may take several minutes.")
    
    sagemaker_client = boto3.client('sagemaker')
    
    # Keep track of job statuses
    job_statuses = {job_name: 'InProgress' for job_name in job_names}
    completed_statuses = {'Completed', 'Failed', 'Stopped'}
    
    # Monitor jobs until all are complete
    while True:
        all_complete = True
        
        for job_name in job_names:
            if job_statuses[job_name] not in completed_statuses:
                try:
                    response = sagemaker_client.describe_processing_job(ProcessingJobName=job_name)
                    current_status = response['ProcessingJobStatus']
                    
                    # Update status if it changed
                    if job_statuses[job_name] != current_status:
                        job_statuses[job_name] = current_status
                        print(f"{job_name}: {current_status}")
                        
                        if current_status == 'Failed':
                            failure_reason = response.get('FailureReason', 'No failure reason provided')
                            print(f"  Failure reason: {failure_reason}")
                        elif current_status == 'Completed':
                            print(f"  ✅ Job completed successfully!")
                    
                    # Check if this job is still running
                    if current_status not in completed_statuses:
                        all_complete = False
                        
                except Exception as e:
                    print(f"Error checking {job_name}: {e}")
                    # Assume job is still running if we can't check it
                    all_complete = False
            
        # If all jobs are complete, break the loop
        if all_complete:
            print("\n🎉 All distillation jobs have completed!")
            break
        
        # Wait before checking again
        print("Checking again in 30 seconds...")
        time.sleep(30)
        clear_output(wait=True)
        print(f"Monitoring {len(job_names)} distillation jobs...")
        
        # Show current status of all jobs
        for job_name, status in job_statuses.items():
            print(f"{job_name}: {status}")
        print("\nWaiting for jobs to complete...")
        
else:
    print("No jobs to monitor.")

## 8. Analyze Distillation Results

Once jobs complete, let's analyze the results and compare model sizes.

In [ ]:
# Analyze distillation results
print("Analyzing distillation results...")

results = []

for job_key, output_path in job_output_paths.items():
    try:
        # Check if distillation info file exists
        info_key = output_path.replace('s3://' + bucket + '/', '') + '/distillation_info.json'
        
        try:
            response = s3_client.get_object(Bucket=bucket, Key=info_key)
            info_content = response['Body'].read().decode('utf-8')
            info_data = json.loads(info_content)
            
            results.append({
                'job_key': job_key,
                'teacher_model': info_data['teacher_model'],
                'student_config': info_data['student_config'],
                'teacher_parameters': info_data['teacher_parameters'],
                'student_parameters': info_data['student_parameters'],
                'size_reduction_percent': info_data['size_reduction_percent'],
                'temperature': info_data['temperature'],
                'alpha': info_data['alpha'],
                'epochs': info_data['epochs'],
                'output_path': output_path
            })
            
            print(f"✅ Distilled model found at {output_path}")
            print(f"   Teacher parameters: {info_data['teacher_parameters']:,}")
            print(f"   Student parameters: {info_data['student_parameters']:,}")
            print(f"   Size reduction: {info_data['size_reduction_percent']:.2f}%")
            
        except s3_client.exceptions.NoSuchKey:
            print(f"⚠️ No distilled model found at {output_path}")
            print(f"   The distillation job for {job_key} may have failed silently.")
            
    except Exception as e:
        print(f"Error checking results for {job_key}: {e}")

if results:
    # Create summary DataFrame
    df = pd.DataFrame(results)
    
    print("\n" + "="*80)
    print("KNOWLEDGE DISTILLATION SUMMARY")
    print("="*80)
    
    for _, row in df.iterrows():
        print(f"Job: {row['job_key']}")
        print(f"  Teacher: {row['teacher_model']}")
        print(f"  Student: {row['student_config']}")
        print(f"  Parameters: {row['teacher_parameters']:,} → {row['student_parameters']:,}")
        print(f"  Size Reduction: {row['size_reduction_percent']:.2f}%")
        print(f"  Training: T={row['temperature']}, α={row['alpha']}, epochs={row['epochs']}")
        print()
    
    # Visualization
    if len(results) > 1:
        plt.figure(figsize=(12, 6))
        
        # Plot 1: Parameter comparison
        plt.subplot(1, 2, 1)
        x_pos = range(len(results))
        teacher_params = [r['teacher_parameters'] for r in results]
        student_params = [r['student_parameters'] for r in results]
        
        plt.bar([x - 0.2 for x in x_pos], teacher_params, 0.4, label='Teacher', alpha=0.8)
        plt.bar([x + 0.2 for x in x_pos], student_params, 0.4, label='Student', alpha=0.8)
        
        plt.xlabel('Model Pairs')
        plt.ylabel('Parameters')
        plt.title('Teacher vs Student Model Sizes')
        plt.xticks(x_pos, [r['job_key'] for r in results], rotation=45)
        plt.legend()
        plt.yscale('log')
        
        # Plot 2: Size reduction
        plt.subplot(1, 2, 2)
        size_reductions = [r['size_reduction_percent'] for r in results]
        plt.bar(range(len(results)), size_reductions, alpha=0.8, color='green')
        
        plt.xlabel('Model Pairs')
        plt.ylabel('Size Reduction (%)')
        plt.title('Model Size Reduction Through Distillation')
        plt.xticks(range(len(results)), [r['job_key'] for r in results], rotation=45)
        
        plt.tight_layout()
        plt.show()

else:
    print("No successful distillation results found.")

## 9. Next Steps

After knowledge distillation, you can:

1. **Deploy the distilled models** for faster inference
2. **Compare performance** between teacher and student models
3. **Fine-tune** the student models further if needed
4. **Combine with other optimization techniques** like quantization or pruning

The distilled models are saved in S3 and ready for deployment or further optimization.